# CustomerAtlas-5D
## K-Means from Scratch on 541K+ Real Online Retail Transactions

This notebook turns raw retail transactions into a **five-dimensional customer embedding** and clusters customers using both:

1. a vectorized **K-Means implementation from scratch**, and
2. scikit-learn's optimized `KMeans` as an independent baseline.

### The five dimensions

| Dimension | Meaning |
|---|---|
| `RecencyDays` | Days since the customer's last purchase |
| `Frequency` | Number of unique orders |
| `Monetary` | Total positive revenue |
| `AvgOrderValue` | Revenue per order |
| `ProductDiversity` | Number of unique products purchased |

The workflow deliberately starts from a messy transactional dataset rather than a ready-made clustering table.

**Dataset:** UCI Online Retail, dataset ID 352  
**DOI:** 10.24432/C5BW33  
**License:** CC BY 4.0


## Project roadmap

```text
541K+ Transactions
       |
       v
Transaction Cleaning
       |
       v
Customer Aggregation
       |
       v
5D Customer Embedding
       |
       v
Outlier Capping + log1p + Standardization
       |
       +-----------------------+
       |                       |
       v                       v
K-Means From Scratch     scikit-learn KMeans
       |                       |
       +-----------+-----------+
                   |
                   v
      Multi-Metric K Selection
                   |
                   v
        Stability + Validation
                   |
          +--------+--------+
          |        |        |
          v        v        v
         PCA    Personas  Anomalies
                   |
                   v
             CSV Artifacts
```


In [ ]:
# Install dependencies when running in a clean notebook/Colab environment.
%pip install -q numpy pandas matplotlib scikit-learn ucimlrepo pytest


In [ ]:
from pathlib import Path
import sys, json, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    silhouette_score,
    normalized_mutual_info_score,
)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.kmeans5d import KMeansScratch
from src.data_pipeline import (
    FIVE_D_FEATURES,
    fetch_uci_online_retail,
    clean_transactions,
    engineer_customer_5d,
    prepare_kmeans_matrix,
)
from src.evaluation import (
    evaluate_k_range,
    choose_k,
    cluster_feature_importance,
)

OUTPUTS = ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Project root:", ROOT)


# 1. Load the real UCI dataset

The first run downloads the dataset through the official `ucimlrepo` interface and caches it under `data/online_retail.csv`.

Subsequent runs use the cache.


In [ ]:
raw = fetch_uci_online_retail(ROOT / "data" / "online_retail.csv")

print("Raw shape:", raw.shape)
display(raw.head())
display(raw.dtypes.to_frame("dtype"))


# 2. Transaction audit

Real transaction data contains phenomena that matter for clustering:

- cancellations / returns,
- non-positive quantities,
- non-positive prices,
- missing customer identifiers in some exports,
- extreme purchases,
- repeated products within an invoice.

We remove records that do not represent positive completed purchases.


In [ ]:
audit = {
    "rows": len(raw),
    "columns": len(raw.columns),
    "missing_customer_id": int(raw["CustomerID"].isna().sum()) if "CustomerID" in raw else None,
    "nonpositive_quantity": int((pd.to_numeric(raw["Quantity"], errors="coerce") <= 0).sum()) if "Quantity" in raw else None,
    "nonpositive_price": int((pd.to_numeric(raw["UnitPrice"], errors="coerce") <= 0).sum()) if "UnitPrice" in raw else None,
}
pd.Series(audit, name="count")


In [ ]:
clean = clean_transactions(raw)

print(f"Rows before cleaning : {len(raw):,}")
print(f"Rows after cleaning  : {len(clean):,}")
print(f"Retained             : {100*len(clean)/len(raw):.2f}%")
display(clean.head())


# 3. Engineer a true 5D customer space

We aggregate hundreds of thousands of transactions into one row per customer.

This step turns the problem from **transaction clustering** into **customer behavior clustering**.


In [ ]:
customers = engineer_customer_5d(clean)

print("Customers:", len(customers))
display(customers.head())
display(customers[FIVE_D_FEATURES].describe().T)


# 4. Why preprocessing matters

K-Means minimizes squared Euclidean distance.

Retail variables are strongly right-skewed, so raw high-spending customers can dominate distance geometry.

We therefore:

1. cap only the extreme upper 0.5% tail,
2. apply `log1p`,
3. standardize every dimension.

This keeps the five dimensions comparable while preserving relative customer structure.


In [ ]:
skew_table = customers[FIVE_D_FEATURES].skew().sort_values(ascending=False).to_frame("raw_skew")
display(skew_table)

fig, axes = plt.subplots(1, 5, figsize=(18, 3.2))
for ax, col in zip(axes, FIVE_D_FEATURES):
    ax.hist(customers[col], bins=50)
    ax.set_title(col)
    ax.set_yscale("log")
plt.suptitle("Raw 5D customer distributions — log y-axis")
plt.tight_layout()
plt.show()


In [ ]:
X, logged, scaler, caps = prepare_kmeans_matrix(customers, quantile_cap=0.995)

processed = pd.DataFrame(X, columns=FIVE_D_FEATURES, index=customers.index)

print("Matrix shape:", X.shape)
print("Finite:", np.isfinite(X).all())
display(caps.to_frame("99.5% cap"))
display(processed.describe().T)


# 5. Build K-Means from scratch

The repository implementation includes:

- K-Means++ initialization
- vectorized squared-distance computation
- Lloyd updates
- empty-cluster recovery
- convergence tolerance
- multiple restarts
- inertia history


In [ ]:
# Small smoke run before model selection.
demo_model = KMeansScratch(
    n_clusters=4,
    n_init=5,
    max_iter=200,
    tol=1e-5,
    random_state=RANDOM_STATE,
)

t0 = time.perf_counter()
demo_labels = demo_model.fit_predict(X)
elapsed = time.perf_counter() - t0

print("Scratch K-Means")
print("iterations :", demo_model.n_iter_)
print("inertia    :", round(demo_model.inertia_, 3))
print("time       :", round(elapsed, 3), "s")

plt.figure(figsize=(7, 4))
plt.plot(demo_model.history_, marker="o")
plt.xlabel("Lloyd iteration")
plt.ylabel("Inertia")
plt.title("Convergence of the best K-Means restart")
plt.grid(alpha=0.25)
plt.show()


# 6. Choose K with more than one metric

There is no ground-truth cluster label.

We compare candidate K values using:

- Inertia
- Silhouette score
- Calinski-Harabasz score
- Davies-Bouldin score
- Repeated-seed clustering stability (mean pairwise ARI)

The final K is selected by rank aggregation rather than one arbitrary criterion.


In [ ]:
metrics = evaluate_k_range(
    X,
    k_values=range(2, 9),
    silhouette_sample=min(3000, len(X)),
    random_state=RANDOM_STATE,
)

BEST_K, ranked = choose_k(metrics)

display(metrics.round(4))
print("Rank-aggregated BEST_K =", BEST_K)
display(ranked.round(4))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].plot(metrics["k"], metrics["inertia"], marker="o")
axes[0,0].set_title("Elbow / Inertia")
axes[0,0].set_xlabel("K")

axes[0,1].plot(metrics["k"], metrics["silhouette"], marker="o")
axes[0,1].set_title("Silhouette — higher is better")
axes[0,1].set_xlabel("K")

axes[1,0].plot(metrics["k"], metrics["davies_bouldin"], marker="o")
axes[1,0].set_title("Davies-Bouldin — lower is better")
axes[1,0].set_xlabel("K")

axes[1,1].plot(metrics["k"], metrics["stability_ari"], marker="o")
axes[1,1].set_title("Repeated-seed stability — higher is better")
axes[1,1].set_xlabel("K")

plt.tight_layout()
plt.show()


# 7. Final model: scratch vs scikit-learn

A strong implementation should agree structurally with a mature library even though cluster numbers themselves may be permuted.

We compare them using **Adjusted Rand Index (ARI)**, which is permutation-invariant.


In [ ]:
scratch = KMeansScratch(
    n_clusters=BEST_K,
    n_init=15,
    max_iter=300,
    tol=1e-5,
    random_state=RANDOM_STATE,
).fit(X)

sk_model = KMeans(
    n_clusters=BEST_K,
    n_init=30,
    random_state=RANDOM_STATE,
)
sk_labels = sk_model.fit_predict(X)

agreement_ari = adjusted_rand_score(scratch.labels_, sk_labels)

print("Scratch inertia :", round(scratch.inertia_, 3))
print("sklearn inertia :", round(sk_model.inertia_, 3))
print("Scratch/sklearn ARI:", round(agreement_ari, 4))


# 8. PCA: turn five dimensions into a map

Clustering is performed in all five standardized dimensions.

PCA is used **only for visualization**, not for fitting the final K-Means.


In [ ]:
pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)

print("Explained variance ratios:", np.round(pca.explained_variance_ratio_, 4))
print("3-PC cumulative variance:", round(pca.explained_variance_ratio_.sum(), 4))

plt.figure(figsize=(9, 7))
for cluster in range(BEST_K):
    mask = scratch.labels_ == cluster
    plt.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        s=22,
        alpha=0.65,
        label=f"Cluster {cluster}",
    )

plt.xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}%)")
plt.ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}%)")
plt.title("CustomerAtlas — 5D clusters projected into PCA space")
plt.legend()
plt.grid(alpha=0.15)
plt.show()


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

for cluster in range(BEST_K):
    mask = scratch.labels_ == cluster
    ax.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        X_pca[mask, 2],
        s=18,
        alpha=0.55,
        label=f"C{cluster}",
    )

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("3D PCA customer constellation")
ax.legend()
plt.show()


# 9. Cluster DNA

We summarize each cluster by its median values in the original business units.

Then we compute a standardized centroid fingerprint to see **which features actually separate the clusters**.


In [ ]:
clustered = customers.copy()
clustered["Cluster"] = scratch.labels_

profile_median = clustered.groupby("Cluster")[FIVE_D_FEATURES].median()
profile_mean = clustered.groupby("Cluster")[FIVE_D_FEATURES].mean()
cluster_sizes = clustered["Cluster"].value_counts().sort_index().rename("Customers")

display(pd.concat([cluster_sizes, profile_median], axis=1).round(2))


In [ ]:
centroid_df = pd.DataFrame(
    scratch.cluster_centers_,
    columns=FIVE_D_FEATURES,
    index=[f"Cluster {i}" for i in range(BEST_K)],
)

plt.figure(figsize=(10, max(3.8, BEST_K*0.75)))
img = plt.imshow(centroid_df.values, aspect="auto", cmap="coolwarm")
plt.colorbar(img, label="Standardized centroid value")
plt.xticks(range(len(FIVE_D_FEATURES)), FIVE_D_FEATURES, rotation=30, ha="right")
plt.yticks(range(BEST_K), centroid_df.index)
plt.title("Cluster DNA — standardized five-dimensional centroids")

for i in range(BEST_K):
    for j in range(len(FIVE_D_FEATURES)):
        plt.text(j, i, f"{centroid_df.iloc[i,j]:.2f}", ha="center", va="center")

plt.tight_layout()
plt.show()


# 10. Which dimensions drive the segmentation?

The score below is the fraction of each feature's total variance explained by separation between cluster means.

It is descriptive — not causal — but useful for interpretation.


In [ ]:
importance = cluster_feature_importance(X, scratch.labels_)
importance_df = (
    pd.DataFrame({"Feature": FIVE_D_FEATURES, "BetweenVarianceRatio": importance})
      .sort_values("BetweenVarianceRatio", ascending=False)
)

display(importance_df.round(4))

plt.figure(figsize=(8, 4))
plt.bar(importance_df["Feature"], importance_df["BetweenVarianceRatio"])
plt.ylabel("Between-cluster / total variance")
plt.title("Features that most distinguish the customer clusters")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


# 11. Cluster personas

This cell converts numeric profiles into concise, reproducible descriptions.

The labels are generated from the cluster statistics rather than manually assigning attractive names.


In [ ]:
global_med = customers[FIVE_D_FEATURES].median()

def persona(row):
    high = [c for c in FIVE_D_FEATURES if row[c] > 1.35 * global_med[c]]
    low = [c for c in FIVE_D_FEATURES if row[c] < 0.70 * global_med[c]]

    parts = []
    if high:
        parts.append("high " + ", ".join(high[:2]))
    if low:
        parts.append("low " + ", ".join(low[:2]))
    return "; ".join(parts) if parts else "balanced / mid-market"

personas = profile_median.apply(persona, axis=1).rename("Persona")
persona_table = pd.concat([cluster_sizes, profile_median, personas], axis=1)

display(persona_table.round(2))


# 12. Rare-customer / anomaly lens

K-Means itself is not an anomaly detector, but distance to the assigned centroid is a useful diagnostic.

We flag the farthest 2.5% of customers in standardized 5D space.


In [ ]:
dist_matrix = scratch.transform(X)
assigned_distance = dist_matrix[np.arange(len(X)), scratch.labels_]

threshold = np.quantile(assigned_distance, 0.975)
clustered["CentroidDistance"] = assigned_distance
clustered["RarePattern"] = assigned_distance >= threshold

print("Distance threshold:", round(float(threshold), 3))
print("Rare-pattern customers:", int(clustered["RarePattern"].sum()))

display(
    clustered[clustered["RarePattern"]]
      .sort_values("CentroidDistance", ascending=False)
      .head(15)
)


# 13. Country as an external validation variable

Country is **not** one of the five clustering dimensions.

We use it only after clustering to ask whether behavioral segments have geographic structure.


In [ ]:
if "Country" in clustered.columns:
    country_cluster = pd.crosstab(
        clustered["Country"],
        clustered["Cluster"],
        normalize="columns"
    )

    top_countries = (
        clustered["Country"]
        .value_counts()
        .head(10)
        .index
    )

    display(country_cluster.loc[country_cluster.index.intersection(top_countries)].round(3))


# 14. Export GitHub-ready artifacts


In [ ]:
metrics.to_csv(OUTPUTS / "k_selection_metrics.csv", index=False)
ranked.to_csv(OUTPUTS / "k_selection_ranked.csv", index=False)
clustered.to_csv(OUTPUTS / "customer_clusters.csv")
persona_table.to_csv(OUTPUTS / "cluster_personas.csv")
centroid_df.to_csv(OUTPUTS / "standardized_centroids.csv")
importance_df.to_csv(OUTPUTS / "feature_importance.csv", index=False)

summary = {
    "dataset": "UCI Online Retail",
    "uci_id": 352,
    "doi": "10.24432/C5BW33",
    "raw_rows": int(len(raw)),
    "clean_rows": int(len(clean)),
    "customers": int(len(customers)),
    "dimensions": FIVE_D_FEATURES,
    "selected_k": int(BEST_K),
    "scratch_inertia": float(scratch.inertia_),
    "sklearn_inertia": float(sk_model.inertia_),
    "scratch_vs_sklearn_ari": float(agreement_ari),
}

(OUTPUTS / "summary.json").write_text(json.dumps(summary, indent=2))

print(json.dumps(summary, indent=2))
print("\nSaved outputs:")
for path in sorted(OUTPUTS.iterdir()):
    print(" -", path.name)


# 15. What to try next

Strong extensions for this repository:

- MiniBatch K-Means for streaming customers
- GPU K-Means with RAPIDS cuML
- GMM for soft memberships
- DBSCAN / HDBSCAN for non-spherical segments
- temporal clustering by month or quarter
- cluster migration analysis
- customer lifetime value modeling
- autoencoder embeddings before clustering
- FAISS K-Means for very large datasets
- compare Euclidean K-Means with cosine/spherical K-Means

---

## Key lesson

The difficult part of real clustering is rarely the `fit()` call.

The real work is:

**data cleaning → representation design → scaling → K selection → stability → interpretation → validation.**
